# 🐄 Train the cow detector

Builds a training set from the public [WAID](https://github.com/xiaohuicui/WAID) aerial dataset,
fine-tunes YOLOv8 on it, and tries the result on one of your own drone photos.

Run the cells top to bottom. Needs `pip install -e ".[notebooks]"` and a clone of WAID
(`git clone https://github.com/xiaohuicui/WAID.git`).

## ⚙️ Settings

In [ ]:
from pathlib import Path

# Work from the repository root, whether Jupyter was started here or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

WAID_PATH = ROOT / "WAID"                  # <-- EDIT: where you cloned WAID
EPOCHS = 20                                # <-- EDIT: 5 for a quick test, 20-80 for a real run
YOUR_PHOTO = ROOT / "photos" / "my_drone_photo.jpg"   # <-- EDIT: a drone photo to test on

print("WAID found:", WAID_PATH.exists())

## 🗂️ Step 1 — Build the dataset

Keeps every WAID image containing cattle and relabels them as a single `cow` class.
Takes a minute or two (it copies a few thousand images).

In [ ]:
from cow_identifier.prepare_data import build_dataset

counts = build_dataset(WAID_PATH, ROOT / "dataset")
print("Images per split:", counts)

## 👀 Step 2 — Sanity check: a few training images with their labels

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

images = list((ROOT / "dataset" / "train" / "images").glob("*.jpg"))
sample = random.sample(images, min(3, len(images)))

fig, axes = plt.subplots(1, len(sample), figsize=(16, 5))
axes = [axes] if len(sample) == 1 else axes
for ax, path in zip(axes, sample):
    img = cv2.imread(str(path))
    h, w = img.shape[:2]
    label = ROOT / "dataset" / "train" / "labels" / f"{path.stem}.txt"
    for line in label.read_text().strip().splitlines():
        _, x, y, bw, bh = map(float, line.split())
        cv2.rectangle(img, (int((x - bw / 2) * w), int((y - bh / 2) * h)),
                           (int((x + bw / 2) * w), int((y + bh / 2) * h)), (0, 255, 0), 2)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.suptitle("WAID training images (green = labelled cows)")
plt.show()

## 🏋️ Step 3 — Train

The slow part. On a laptop CPU, 5 epochs took about 4 hours; a GPU (or `device="mps"` on Apple
Silicon) is much faster. Training can also be run from a terminal with `cow-train`.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")      # COCO-pretrained starting point
model.train(
    data=str(ROOT / "dataset" / "data.yaml"),
    epochs=EPOCHS, imgsz=640, batch=8, name="cow_model", patience=15,
    # device="mps",             # <-- uncomment on Apple Silicon
)

BEST = ROOT / "runs" / "detect" / "cow_model" / "weights" / "best.pt"
print("Trained model saved at:", BEST)

## 🎯 Step 4 — Try it on your photo

Left: the plain COCO model. Right: your cow model. Tiled detection (as in `cow-detect`) is used so
small cows in a large drone photo are not lost.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from cow_identifier.detect import detect_on_image, draw_detections

img = cv2.imread(str(YOUR_PHOTO))
assert img is not None, f"Could not read {YOUR_PHOTO}"

plain = detect_on_image(YOLO("yolov8s.pt"), img, conf_thresh=0.25)
cows = detect_on_image(YOLO(str(BEST)), img, conf_thresh=0.25)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, dets, title in [(axes[0], plain, "Before — plain COCO model"), (axes[1], cows, "After — cow model")]:
    ax.imshow(cv2.cvtColor(draw_detections(img, dets), cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title}: {len(dets)} detection(s)"); ax.axis("off")
plt.show()